# Setup — tạo resource

Notebook này **chỉ tạo hạ tầng**, không xử lý dữ liệu và không xóa dữ liệu:

| Bước | Hàm | Tạo ra |
|---|---|---|
| 1 | `archive.provision()` | S3 bucket, IAM roles, DMS subnet group, S3 Gateway Endpoint, replication instance, source/target endpoints, replication task ở trạng thái `ready` |
| 2 | `partition_initial.setup_partition_job()` | Glue job, Glue crawler, Glue database, IAM role cho bước repartition |

Chạy pipeline (full load, repartition, verify, purge) nằm ở `run_pipeline.ipynb`.

Trước khi chạy: `dms/.env` không còn giá trị `<...>`, `aws sts get-caller-identity` trả về đúng account, RDS `available` và security group của RDS cho phép security group của DMS vào port PostgreSQL.

In [6]:
%pip install boto3 python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


In [7]:
import importlib.util
import sys
from pathlib import Path

notebook_dir = Path.cwd()
if not (notebook_dir / 'archive.py').is_file():
    notebook_dir = Path.cwd() / 'dms'


def load(name):
    """Nạp lại module từ ổ đĩa để không dùng bản cũ còn trong kernel."""
    module_path = (notebook_dir / f'{name}.py').resolve()
    sys.modules.pop(name, None)
    spec = importlib.util.spec_from_file_location(name, module_path)
    if spec is None or spec.loader is None:
        raise ImportError(f'Không thể load module từ {module_path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    print(f'Loaded: {module.__file__}')
    return module


archive = load('archive')
partition_initial = load('partition_initial')

cfg = archive.config()
print(f'Region  : {cfg.aws_region}')
print(f'Nguồn   : {cfg.source_schema}.{cfg.source_table} ({cfg.date_column} cũ hơn {cfg.retention_days} ngày)')
print(f'Raw     : s3://{cfg.s3_bucket}/{cfg.s3_prefix}/')
print(f'DMS     : {cfg.instance_id}, {cfg.task_id}')

Loaded: D:\Project để phỏng vấn\Archive\Archiver-Data\dms\archive.py
Loaded: D:\Project để phỏng vấn\Archive\Archiver-Data\dms\partition_initial.py
Region  : ap-southeast-1
Nguồn   : public.orders (closed_at_utc cũ hơn 90 ngày)
Raw     : s3://my-data-lake-archival-demo/raw/rds/orders/
DMS     : orders-archive-instance, orders-archive-task


## 1. Hạ tầng DMS

`provision()` là idempotent: gọi lại sẽ tái sử dụng resource đã có. Hàm test cả hai endpoint và dừng ngay nếu RDS hoặc S3 không kết nối được. Bước tạo replication instance thường mất vài phút — đừng đóng kernel giữa lúc chờ.

Hàm này **không** start full load, nên chạy lại an toàn.

In [8]:
archive.provision()

PROVISION — hạ tầng DMS cho PostgreSQL RDS → S3
Nguồn : public.orders
Bộ lọc: closed_at_utc cũ hơn 90 ngày
Đích  : s3://my-data-lake-archival-demo/raw/rds/orders/
✓ S3 bucket đã tồn tại: my-data-lake-archival-demo
✓ IAM role cho VPC: dms-vpc-role
✓ IAM role cho S3: orders-archive-s3-role
✓ Subnet group đã tồn tại: orders-archive-subnet
✓ Private route tới S3: vpce-0a8afdf2e593f39da
✓ DMS instance đã tồn tại: orders-archive-instance
✓ Source endpoint đã tồn tại: orders-archive-source
✓ Target endpoint đã tồn tại: orders-archive-target
  Bắt đầu test kết nối RDS
  Bắt đầu test kết nối S3
  Đang test endpoint: {'RDS': 'testing', 'S3': 'testing'}
  Đang test endpoint: {'RDS': 'testing', 'S3': 'testing'}
  Đang test endpoint: {'RDS': 'testing', 'S3': 'testing'}
  Đang test endpoint: {'RDS': 'testing', 'S3': 'successful'}
  Đang test endpoint: {'RDS': 'testing', 'S3': 'successful'}
  Đang test endpoint: {'RDS': 'testing', 'S3': 'successful'}
  Đang test endpoint: {'RDS': 'testing', 'S3': 'su

## 2. Glue job repartition + crawler

Upload Glue script, tạo job, crawler, Glue database và IAM role. Chưa xử lý dữ liệu. Cần `CURATED_PREFIX`, `GLUE_DATABASE`, `GLUE_TABLE_PREFIX` trong `dms/.env`.

In [9]:
partition_initial.setup_partition_job()

Glue job ready: orders-archive-partition-initial
Source : s3://my-data-lake-archival-demo/raw/rds/orders/public/orders/
Target : s3://my-data-lake-archival-demo/curated/rds/orders/year=YYYY/month=MM/day=DD/


## 3. Teardown

Chỉ chạy khi đã kiểm tra xong dữ liệu trên S3. Cả hai hàm đều giữ nguyên RDS, S3 bucket, dữ liệu raw/curated, Glue database và catalog tables.

In [ ]:
# archive.destroy()                            # xóa DMS instance, endpoints, task, subnet group, IAM role
# partition_initial.destroy_partition_job()    # xóa Glue job, crawler, IAM role của bước repartition

DESTROY — chỉ xóa tài nguyên DMS
  Đang chờ task được xóa...
  Đang chờ task được xóa...
✓ Đã xóa task: orders-archive-task
  Đang chờ endpoint orders-archive-source được xóa...
  Đang chờ endpoint orders-archive-source được xóa...
  Đang chờ endpoint orders-archive-source được xóa...
  Đang chờ endpoint orders-archive-source được xóa...
  Đang chờ endpoint orders-archive-source được xóa...
  Đang chờ endpoint orders-archive-source được xóa...
  Đang chờ endpoint orders-archive-source được xóa...
  Đang chờ endpoint orders-archive-source được xóa...
  Đang chờ endpoint orders-archive-source được xóa...
  Đang chờ endpoint orders-archive-source được xóa...
  Đang chờ endpoint orders-archive-source được xóa...
  Đang chờ endpoint orders-archive-source được xóa...
  Đang chờ endpoint orders-archive-source được xóa...
  Đang chờ endpoint orders-archive-source được xóa...
  Đang chờ endpoint orders-archive-source được xóa...
  Đang chờ endpoint orders-archive-source được xóa...
  Đang chờ e